# BISVEDA Voice Call Assistant — Twilio + Gemini Live

**AI-powered multilingual voice interface for the Bureau of Indian Standards**
(बिसवेदा — BIS + Veda)  
Problem Statement: SIH26107 | Team: fsociety

This notebook implements a **real-time voice call assistant** that lets users
query BIS standards, verify ISI marks, check HUID, assess QCO compliance,
find testing labs, and more — all through a natural phone conversation in
their preferred Indian language.

> **Why Twilio instead of Exotel?** The original prototype used Exotel's
> Voice Streaming (Connect Voicebot Applet), which is a paid product and
> isn't provisioned for the BISVEDA project. Twilio offers a **free trial**
> with credit that comfortably covers hackathon demo calls, and its Media
> Streams feature does the same job (bidirectional call audio over a
> WebSocket). Everything downstream of the audio transport — Gemini Live,
> tool calling, the mock BISVEDA backend, and logging — is unchanged.

```text
User (phone call)
  |
Twilio Media Streams (bidirectional <Connect><Stream> WebSocket, 8kHz mu-law)
  |
wss://<ngrok>/bisveda/media
  |
FastAPI WebSocket bridge  ── structured logging (console + JSON file)
  | / ^
Gemini Live (audio in/out + FUNCTION CALLING)
  |         |
  |   Tool dispatch → BISVEDA backend
  |     ├── search_standards()
  |     ├── verify_isi_mark()
  |     ├── verify_huid()
  |     ├── check_qco_compliance()
  |     ├── find_testing_labs()
  |     ├── get_standard_passport()
  |     ├── report_counterfeit()
  |     ├── file_complaint()
  |     └── end_call()
  |
SQLite: call_sessions + query_log
```

### How the voice flow works
1. User calls in (or receives a call via `call_bisveda()`, which uses the Twilio REST API)
2. BISVEDA greets in Hindi, invites the user to speak in any language
3. Gemini Live detects the language from the user's speech and continues in it
4. User asks their query naturally — "Mera ISI mark check karna hai",
   "इस प्रोडक्ट का QCO status बताओ", "What is IS 1239?"
5. Gemini calls the appropriate BISVEDA tool (search, verify, check)
6. Results are spoken back to the user in their language
7. Multi-turn: user can ask follow-ups, switch topics, or end the call

### What changed vs. the Exotel/Kaggle version
| Concern | Exotel (old) | Twilio (this notebook) |
|---|---|---|
| Outbound call | `EXOTEL_API_BASE/.../Calls/connect` REST call | `twilio_client.calls.create(...)` (Twilio Python SDK) |
| Stream setup | `StreamUrl` passed in the connect API call | TwiML `<Connect><Stream url="..."/></Connect>` returned to Twilio |
| WS start event | `event.start.call_sid` / `stream_sid` | `event.start.callSid` / `event.start.streamSid` (camelCase) |
| WS media event | raw 8kHz **PCM16**, base64 | 8kHz **mu-law**, base64 (`audio/x-mulaw`) — needs mu-law⇄PCM16 conversion |
| Audio out | 8kHz PCM16 → Exotel | PCM16 → mu-law → Twilio |
| Barge-in | not handled | Twilio `clear` event sent when Gemini reports `interrupted`, so BISVEDA's audio stops instantly when the caller starts talking over it |
| Cost | Paid, not provisioned | Free trial credit (~$15), enough for demo calls |

### Architecture alignment with BISVEDA
- **Voice Lifecycle**: Gemini Live handles ASR + TTS natively; for production,
  Bhashini ASR/TTS (with IndicConformer/IndicTTS fallback) would replace this
  for 22+ scheduled language coverage and air-gapped deployment.
- **Multilingual**: Gemini Live supports Hindi, English, and other Indian
  languages natively; Bhashini integration extends to all 22 scheduled languages.
- **Tool Calling**: Maps directly to BISVEDA's use cases — each tool corresponds
  to a Consumer/MSME use case from the architecture.
- **Incremental Logging**: Every query, tool result, and transcript is logged
  for analytics and audit.


In [ ]:
# 1. Install dependencies
!pip -q install -U google-genai fastapi uvicorn pyngrok requests websockets twilio


In [ ]:
# 2. Configuration
#
# Loads API keys from Kaggle Secrets, Colab Userdata, or Environment Variables.
# Gemini 3.1 Flash Live handles real-time audio + function calling.
# Twilio handles telephony over bidirectional Media Streams WebSocket.

import os

def _get_secret(key, default=""):
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(key)
        if val:
            return str(val)
    except Exception:
        pass
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return str(val)
    except Exception:
        pass
    return os.environ.get(key, default)

# ── Gemini ──
GEMINI_API_KEY   = _get_secret("VOICE_GEMINI_API_KEY") or _get_secret("GEMINI_API_KEY", "AQ.Ab8RN6JrOHqgvRTeVBWciSg1P4LcTEuVq6lDEh7tF0EctC-1Mw")
GEMINI_MODEL     = "gemini-3.1-flash-live-preview"

# ── Twilio ──
TWILIO_ACCOUNT_SID  = _get_secret("TWILIO_ACCOUNT_SID", "ACd2780ca61739d2d2420b2447e1d5a7dd")
TWILIO_AUTH_TOKEN   = _get_secret("TWILIO_AUTH_TOKEN", "609db245347d7dd11bda0ab7c5ae29cf")
TWILIO_PHONE_NUMBER = _get_secret("TWILIO_PHONE_NUMBER", "+17372508034")
MY_PHONE_NUMBER     = _get_secret("MY_PHONE_NUMBER", "9529512485")

# ── Server / tunnel ──
PORT            = 8001
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN", "3JuYjB10Je24Y4vDMc7nmD8yvqk_6EvETNmVk772rHuTkJVLV")
TWILIO_SAMPLE_RATE = 8000

# ── Work directory & Database ──
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(WORK_DIR, exist_ok=True)
DB_PATH        = os.path.join(WORK_DIR, "bisveda_voice.db")
LOG_FILE       = os.path.join(WORK_DIR, "bisveda_voice_call.log")
LOG_JSONL_FILE = os.path.join(WORK_DIR, "bisveda_voice_call.jsonl")

print("Configuration loaded.")
print(f"Gemini model:    {GEMINI_MODEL}")
print(f"Twilio SID:      {TWILIO_ACCOUNT_SID}")
print(f"Twilio number:   {TWILIO_PHONE_NUMBER}")
print(f"Calling number:  {MY_PHONE_NUMBER}")
print(f"Database:        {DB_PATH}")


In [ ]:
# 3. Imports + structured logging
#
# Every call, tool invocation, DB write and error gets a "stage" tag
# so post-call diagnostics can grep by stage (TOOL_ERROR, GEMINI_ERROR, etc.).
import asyncio
import base64
import json
import logging
import audioop
import sqlite3
import threading
import time
import traceback
from urllib.parse import urlencode
import requests
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Response, Request
from google import genai
from google.genai import types
from twilio.rest import Client as TwilioClient
from twilio.twiml.voice_response import VoiceResponse, Connect
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("bisveda-voice")
with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("BISVEDA VOICE CALL ASSISTANT — CALL LOG (Twilio)\n")
    f.write("=" * 80 + "\n")
open(LOG_JSONL_FILE, "w", encoding="utf-8").close()
class StructuredLogger:
    """Human-readable lines to console/LOG_FILE + one JSON object per line
    to LOG_JSONL_FILE for programmatic filtering by call_id / stage."""
    def __init__(self, text_path, jsonl_path):
        self.text_path = text_path
        self.jsonl_path = jsonl_path
        self._lock = threading.Lock()
    def _write(self, level, stage, message, call_id=None, extra=None):
        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        line = (
            f"[{ts}] [{level}] [{stage}]"
            + (f" [call={call_id}]" if call_id else "")
            + f" {message}"
        )
        record = {
            "ts": ts, "level": level, "stage": stage,
            "call_id": call_id, "message": message,
        }
        if extra:
            record["extra"] = extra
        with self._lock:
            print(line, flush=True)
            with open(self.text_path, "a", encoding="utf-8") as f:
                f.write(line + "\n")
            with open(self.jsonl_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
    def info(self, stage, message, call_id=None, extra=None):
        self._write("INFO", stage, message, call_id, extra)
    def warn(self, stage, message, call_id=None, extra=None):
        self._write("WARN", stage, message, call_id, extra)
    def error(self, stage, message, call_id=None, extra=None):
        self._write("ERROR", stage, message, call_id, extra)
logger = StructuredLogger(LOG_FILE, LOG_JSONL_FILE)
print("Structured logger ready ->  ", LOG_FILE)


In [ ]:
# 4. Database layer — call_sessions + query_log
#
#   call_sessions : one row per voice call (caller info, language, duration)
#   query_log     : one row per USER QUERY within a call (what they asked,
#                   which tool was invoked, what was returned). A single call
#                   can have multiple queries (multi-turn conversation).
#
# Incremental writes: every tool call UPSERTs immediately, so a dropped call
# still retains all queries answered up to that point.

def get_db_connection():
    conn = sqlite3.connect(DB_PATH, timeout=10, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db_connection()
    try:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS call_sessions (
                call_id           TEXT PRIMARY KEY,
                phone_number      TEXT,
                detected_language TEXT,
                call_started_at   TEXT,
                call_ended_at     TEXT,
                duration_seconds  REAL,
                total_queries     INTEGER DEFAULT 0,
                status            TEXT DEFAULT 'in_progress',
                end_reason        TEXT,
                raw_transcript    TEXT,
                satisfaction_score TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS query_log (
                query_id         INTEGER PRIMARY KEY AUTOINCREMENT,
                call_id          TEXT NOT NULL,
                timestamp        TEXT,
                user_query_text  TEXT,
                detected_intent  TEXT,
                tool_called      TEXT,
                tool_args        TEXT,
                tool_result      TEXT,
                response_summary TEXT,
                language         TEXT,
                FOREIGN KEY (call_id) REFERENCES call_sessions(call_id)
            )
        """)
        conn.commit()
        logger.info("DB_INIT", f"Database ready at {DB_PATH}")
    except Exception as e:
        logger.error("DB_ERROR", f"init_db failed: {type(e).__name__}: {e}")
        raise
    finally:
        conn.close()


def ensure_call_session(call_id, phone_number):
    """Create the call_sessions row at call start."""
    conn = get_db_connection()
    try:
        conn.execute(
            """INSERT OR IGNORE INTO call_sessions
               (call_id, phone_number, call_started_at, status)
               VALUES (?, ?, ?, 'in_progress')""",
            (call_id, phone_number, time.strftime("%Y-%m-%d %H:%M:%S")),
        )
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"ensure_call_session failed: {e}", call_id)
    finally:
        conn.close()


def update_call_session(call_id, **fields):
    """Update call session fields (language, status, etc.)."""
    allowed = {
        "detected_language", "call_ended_at", "duration_seconds",
        "total_queries", "status", "end_reason", "satisfaction_score",
    }
    clean = {k: v for k, v in fields.items() if k in allowed and v is not None}
    if not clean:
        return
    set_clause = ", ".join(f"{k} = ?" for k in clean)
    values = list(clean.values()) + [call_id]
    conn = get_db_connection()
    try:
        conn.execute(f"UPDATE call_sessions SET {set_clause} WHERE call_id = ?", values)
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"update_call_session failed: {e}", call_id)
    finally:
        conn.close()


def log_query(call_id, user_query_text, detected_intent, tool_called,
              tool_args, tool_result, response_summary, language=None):
    """Log each user query and the tool result within a call."""
    conn = get_db_connection()
    try:
        conn.execute(
            """INSERT INTO query_log
               (call_id, timestamp, user_query_text, detected_intent,
                tool_called, tool_args, tool_result, response_summary, language)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (
                call_id, time.strftime("%Y-%m-%d %H:%M:%S"),
                user_query_text, detected_intent, tool_called,
                json.dumps(tool_args, ensure_ascii=False) if tool_args else None,
                json.dumps(tool_result, ensure_ascii=False) if tool_result else None,
                response_summary, language,
            ),
        )
        # Increment query count
        conn.execute(
            "UPDATE call_sessions SET total_queries = total_queries + 1 WHERE call_id = ?",
            (call_id,),
        )
        conn.commit()
        logger.info("QUERY_LOGGED", f"intent={detected_intent} tool={tool_called}", call_id)
    except Exception as e:
        logger.error("DB_ERROR", f"log_query failed: {e}", call_id)
    finally:
        conn.close()


def append_transcript(call_id, speaker, text):
    conn = get_db_connection()
    try:
        row = conn.execute(
            "SELECT raw_transcript FROM call_sessions WHERE call_id = ?", (call_id,)
        ).fetchone()
        existing = (row["raw_transcript"] if row and row["raw_transcript"] else "")
        line = f"[{speaker}] {text}\n"
        conn.execute(
            "UPDATE call_sessions SET raw_transcript = ? WHERE call_id = ?",
            (existing + line, call_id),
        )
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"append_transcript failed: {e}", call_id)
    finally:
        conn.close()


def fetch_all_sessions():
    conn = get_db_connection()
    try:
        rows = conn.execute(
            "SELECT * FROM call_sessions ORDER BY call_started_at DESC"
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


def fetch_all_queries():
    conn = get_db_connection()
    try:
        rows = conn.execute(
            """SELECT q.*, s.phone_number, s.detected_language
               FROM query_log q
               LEFT JOIN call_sessions s ON s.call_id = q.call_id
               ORDER BY q.timestamp DESC"""
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


init_db()
print("BISVEDA voice database initialised.")


In [ ]:
# 5. Mock BISVEDA backend services
#
# In production these would call the real BISVEDA microservices (Standards
# RAG pipeline, ISI/HUID verification APIs, QCO compliance checker, etc.).
# Here they return realistic mock data so the voice flow can be tested
# end-to-end without the full backend.
#
# Replace each function body with actual API calls when integrating.

MOCK_STANDARDS = {
    "IS 1239": {
        "number": "IS 1239",
        "title": "Mild Steel Tubes, Tubulars and Other Wrought Steel Fittings",
        "year": "2004 (Reaffirmed 2019)",
        "scope": "Covers requirements for mild steel tubes used in water, gas, and steam applications",
        "qco_status": "Under QCO — mandatory BIS certification required",
        "certification_scheme": "Scheme-I (ISI Mark)",
    },
    "IS 10500": {
        "number": "IS 10500",
        "title": "Drinking Water — Specification",
        "year": "2012 (Second Revision)",
        "scope": "Specifies requirements for drinking water quality parameters",
        "qco_status": "Referenced in QCOs for packaged drinking water (IS 14543)",
        "certification_scheme": "Scheme-I (ISI Mark) for packaged water",
    },
    "IS 616": {
        "number": "IS 616",
        "title": "Safety Requirements for Electrical Equipment",
        "year": "2017",
        "scope": "General requirements for safety of household and similar electrical appliances",
        "qco_status": "Under QCO — 48 categories of electronics under compulsory registration",
        "certification_scheme": "Scheme-VII (CRS) for electronics",
    },
    "IS 2062": {
        "number": "IS 2062",
        "title": "Hot Rolled Medium and High Tensile Structural Steel",
        "year": "2011",
        "scope": "Covers hot-rolled structural steel plates, strips, shapes and sections",
        "qco_status": "Under QCO — mandatory BIS certification",
        "certification_scheme": "Scheme-I (ISI Mark)",
    },
}

MOCK_ISI_MARKS = {
    "CM/L-1234567": {
        "valid": True,
        "manufacturer": "Demo Steel Industries Pvt. Ltd.",
        "product": "Mild Steel Tubes (IS 1239)",
        "licence_valid_until": "2027-03-31",
        "status": "Active",
    },
    "CM/L-9999999": {
        "valid": False,
        "status": "Not found in BIS records — possibly counterfeit",
    },
}

MOCK_HUIDS = {
    "AB1234CD": {
        "valid": True,
        "item_type": "Gold Ring — 22 Karat",
        "hallmarking_centre": "Mumbai AHC-127",
        "purity": "22K (916)",
        "date_hallmarked": "2026-05-14",
    },
}

MOCK_LABS = [
    {
        "name": "BIS Testing Lab — Mumbai",
        "address": "Manakalaya, E9, MIDC, Andheri East, Mumbai 400093",
        "scope": "Steel, Cement, Electrical, Food Products",
        "phone": "022-28262720",
    },
    {
        "name": "National Test House — Kolkata",
        "address": "Block CP, Sector V, Salt Lake, Kolkata 700091",
        "scope": "Metals, Chemicals, Textiles, Rubber",
        "phone": "033-23675088",
    },
    {
        "name": "STQC Lab — Delhi",
        "address": "Electronics Niketan, CGO Complex, New Delhi 110003",
        "scope": "Electronics, IT Products, Telecom Equipment",
        "phone": "011-24363083",
    },
]


def backend_search_standards(query_text, max_results=3):
    """Search BIS standards by keyword. In production: RAG pipeline query."""
    query_lower = query_text.lower()
    results = []
    for key, std in MOCK_STANDARDS.items():
        if (query_lower in key.lower()
            or query_lower in std["title"].lower()
            or query_lower in std["scope"].lower()):
            results.append(std)
    if not results:
        # Fuzzy fallback: return all if no match
        results = list(MOCK_STANDARDS.values())[:max_results]
    return results[:max_results]


def backend_verify_isi_mark(cm_l_number):
    """Verify an ISI CM/L number. In production: BIS licence API."""
    clean = cm_l_number.strip().upper().replace(" ", "")
    # Normalize formats like "CML1234567" to "CM/L-1234567"
    if clean.startswith("CML") and "/" not in clean:
        clean = "CM/L-" + clean[3:]
    elif clean.startswith("CM/L") and "-" not in clean:
        clean = clean[:4] + "-" + clean[4:]
    return MOCK_ISI_MARKS.get(clean, {
        "valid": False,
        "status": f"CM/L number '{cm_l_number}' not found in BIS records",
    })


def backend_verify_huid(huid):
    """Verify a Hallmark Unique ID. In production: BIS HUID API."""
    clean = huid.strip().upper().replace(" ", "")
    return MOCK_HUIDS.get(clean, {
        "valid": False,
        "status": f"HUID '{huid}' not found — may be invalid or not yet registered",
    })


def backend_check_qco(product_name):
    """Check if a product falls under QCO. In production: QCO database query."""
    product_lower = product_name.lower()
    qco_products = {
        "steel": {"covered": True, "standard": "IS 2062 / IS 1239",
                  "scheme": "Scheme-I (ISI Mark)",
                  "note": "All structural steel and tubes require mandatory BIS certification"},
        "cement": {"covered": True, "standard": "IS 269 / IS 8112 / IS 12269",
                   "scheme": "Scheme-I (ISI Mark)",
                   "note": "All grades of cement are under QCO"},
        "helmet": {"covered": True, "standard": "IS 4151",
                   "scheme": "Scheme-I (ISI Mark)",
                   "note": "Protective helmets for two-wheeler riders"},
        "electronics": {"covered": True, "standard": "Multiple (CRS)",
                        "scheme": "Scheme-VII (CRS Registration)",
                        "note": "48 categories under compulsory registration"},
        "toy": {"covered": True, "standard": "IS 9873 / IS 15644",
                "scheme": "Scheme-I (ISI Mark)",
                "note": "All toys for children under 14 years"},
        "water": {"covered": True, "standard": "IS 14543",
                  "scheme": "Scheme-I (ISI Mark)",
                  "note": "Packaged drinking water and mineral water"},
    }
    for keyword, info in qco_products.items():
        if keyword in product_lower:
            return info
    return {
        "covered": False,
        "note": f"'{product_name}' does not appear to be under a current QCO. "
                "However, new QCOs are notified regularly — check the latest "
                "BIS QCO orders at bis.gov.in for the most current list.",
    }


def backend_find_labs(product_or_standard, location=None):
    """Find BIS-recognized testing labs. In production: lab directory API."""
    results = MOCK_LABS
    if location:
        loc_lower = location.lower()
        filtered = [l for l in results if loc_lower in l["address"].lower()]
        if filtered:
            results = filtered
    return results[:3]


print("Mock BISVEDA backend services ready.")
print(f"  Standards in DB: {len(MOCK_STANDARDS)}")
print(f"  ISI marks:       {len(MOCK_ISI_MARKS)}")
print(f"  HUIDs:           {len(MOCK_HUIDS)}")
print(f"  Labs:            {len(MOCK_LABS)}")


In [ ]:
# 6. Audio helpers — Twilio (8 kHz mu-law) <-> Gemini Live (16 kHz PCM16 in / 24 kHz PCM16 out)
#
# KEY DIFFERENCE FROM EXOTEL: Twilio Media Streams always send and expect
# 8 kHz, 8-bit **mu-law** (audio/x-mulaw) audio, base64-encoded — never raw
# PCM16. So there's an extra mu-law <-> PCM16 conversion step (via audioop)
# on top of the sample-rate conversion.
#
#   Inbound:  Twilio mu-law 8kHz -> PCM16 8kHz -> PCM16 16kHz -> Gemini
#   Outbound: Gemini PCM16 24kHz -> PCM16 8kHz -> mu-law 8kHz -> Twilio

AUDIO_STATE = {}


def twilio_to_gemini_pcm16(payload_b64: str, call_id: str) -> bytes:
    """Twilio 8 kHz mu-law (base64) -> Gemini 16 kHz PCM16 mono."""
    raw_mulaw = base64.b64decode(payload_b64)
    pcm8k = audioop.ulaw2lin(raw_mulaw, 2)  # mu-law -> 16-bit linear PCM, still 8kHz
    state = AUDIO_STATE.setdefault(call_id, {"in": None, "out": None})
    pcm16k, state["in"] = audioop.ratecv(pcm8k, 2, 1, 8000, 16000, state["in"])
    return pcm16k


def gemini_to_twilio_payload(pcm24k: bytes, call_id: str) -> str:
    """Gemini 24 kHz PCM16 -> Twilio 8 kHz mu-law, base64-encoded."""
    state = AUDIO_STATE.setdefault(call_id, {"in": None, "out": None})
    pcm8k, state["out"] = audioop.ratecv(pcm24k, 2, 1, 24000, 8000, state["out"])
    mulaw = audioop.lin2ulaw(pcm8k, 2)
    return base64.b64encode(mulaw).decode("ascii")


def clear_audio_state(call_id: str):
    AUDIO_STATE.pop(call_id, None)


print("Audio helpers ready (mu-law <-> PCM16, 8 kHz <-> 16/24 kHz resampling).")


In [ ]:
# 7. Gemini tool declarations + BISVEDA system prompt
#
# The model gets NINE tools, mapping to BISVEDA's Consumer/MSME use cases:
#   search_standards       — search the BIS standards database
#   verify_isi_mark        — verify an ISI CM/L licence number
#   verify_huid            — verify a Hallmark Unique ID
#   check_qco_compliance   — check if a product is under QCO
#   find_testing_labs      — find BIS-recognized labs / AHCs
#   get_standard_passport  — get full details of a specific standard
#   report_counterfeit     — log a counterfeit / suspicious product report
#   file_complaint         — log a consumer complaint
#   end_call               — end the phone call
#
# None of this changes for Twilio — tool calling happens inside the Gemini
# Live session regardless of which telephony provider carries the audio.

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# In-memory per-call state
CALLS = {}  # call_id -> {"phone": ..., "language": ..., "queries": [...]}


TOOLS = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name="search_standards",
            description=(
                "Search the BIS standards database by keywords, product name, "
                "standard number, or industry/sector. Returns matching "
                "standards with their number, title, scope, QCO status, and "
                "applicable certification scheme. Call this when the user "
                "asks about any Indian Standard, wants to know which "
                "standard applies to a product, or asks general standards "
                "questions."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": (
                            "Search query — a standard number (e.g. 'IS 1239'), "
                            "product name (e.g. 'steel tubes'), or keywords "
                            "(e.g. 'drinking water specification'). Translate "
                            "the user's query to English keywords for search."
                        ),
                    },
                },
                "required": ["query"],
            },
        ),
        types.FunctionDeclaration(
            name="verify_isi_mark",
            description=(
                "Verify whether an ISI Mark CM/L licence number is genuine "
                "and currently active. The user may say it as 'CM/L number', "
                "'ISI number', 'licence number', or spell out digits. Extract "
                "and normalize the number before calling. Returns the "
                "manufacturer name, product, licence validity, and status."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "cm_l_number": {
                        "type": "STRING",
                        "description": "The CM/L licence number, e.g. 'CM/L-1234567'",
                    },
                },
                "required": ["cm_l_number"],
            },
        ),
        types.FunctionDeclaration(
            name="verify_huid",
            description=(
                "Verify a Hallmark Unique Identification (HUID) number "
                "engraved on hallmarked gold/silver jewellery. Returns "
                "item type, purity, hallmarking centre, and date. The user "
                "may spell out the alphanumeric code."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "huid": {
                        "type": "STRING",
                        "description": "The HUID code, e.g. 'AB1234CD'",
                    },
                },
                "required": ["huid"],
            },
        ),
        types.FunctionDeclaration(
            name="check_qco_compliance",
            description=(
                "Check whether a product or product category falls under "
                "a Quality Control Order (QCO), meaning BIS certification "
                "is mandatory before it can be manufactured or imported. "
                "Returns the applicable standard, certification scheme, "
                "and whether it is currently covered."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_name": {
                        "type": "STRING",
                        "description": (
                            "Product name or category in English, e.g. "
                            "'steel pipes', 'cement', 'electronic toys'"
                        ),
                    },
                },
                "required": ["product_name"],
            },
        ),
        types.FunctionDeclaration(
            name="find_testing_labs",
            description=(
                "Find BIS-recognized testing laboratories or Assaying & "
                "Hallmarking Centres (AHCs) for a given product or standard. "
                "Optionally filter by city/state."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_or_standard": {
                        "type": "STRING",
                        "description": "Product name or standard number to find labs for",
                    },
                    "location": {
                        "type": "STRING",
                        "description": "City or state to filter by (optional)",
                    },
                },
                "required": ["product_or_standard"],
            },
        ),
        types.FunctionDeclaration(
            name="get_standard_passport",
            description=(
                "Get the full 'Standard Passport' — a complete summary of "
                "a specific Indian Standard including its number, title, "
                "year, scope, QCO status, applicable certification scheme, "
                "and related standards. Use when the user asks for details "
                "about a specific standard they already know."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "standard_number": {
                        "type": "STRING",
                        "description": "The IS number, e.g. 'IS 1239', 'IS 10500'",
                    },
                },
                "required": ["standard_number"],
            },
        ),
        types.FunctionDeclaration(
            name="report_counterfeit",
            description=(
                "Log a report of a suspected counterfeit product bearing "
                "a fake ISI mark or HUID. Captures product description, "
                "location, and any identifying numbers. Tell the user their "
                "report has been logged and will be forwarded to the "
                "appropriate BIS regional office."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_description": {
                        "type": "STRING",
                        "description": "What the product is (e.g. 'steel pipes with ISI mark')",
                    },
                    "location": {
                        "type": "STRING",
                        "description": "Where the product was found (shop, city)",
                    },
                    "suspicious_number": {
                        "type": "STRING",
                        "description": "The CM/L or HUID number on the suspect product",
                    },
                    "additional_details": {
                        "type": "STRING",
                        "description": "Any other details the caller provided",
                    },
                },
                "required": ["product_description"],
            },
        ),
        types.FunctionDeclaration(
            name="file_complaint",
            description=(
                "Log a consumer complaint about a BIS-certified product "
                "that does not meet the standard's quality. Captures the "
                "complaint details and tells the user it will be forwarded "
                "to the BIS complaint cell."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_description": {
                        "type": "STRING",
                        "description": "Product with the quality issue",
                    },
                    "complaint_details": {
                        "type": "STRING",
                        "description": "What the issue is — broken, substandard, unsafe, etc.",
                    },
                    "cm_l_number": {
                        "type": "STRING",
                        "description": "CM/L number on the product, if available",
                    },
                    "purchase_location": {
                        "type": "STRING",
                        "description": "Where/when the product was bought",
                    },
                },
                "required": ["product_description", "complaint_details"],
            },
        ),
        types.FunctionDeclaration(
            name="end_call",
            description=(
                "Call this ONLY after you have already spoken a polite "
                "closing goodbye out loud. This ends the phone call. "
                "Always thank the user for contacting BISVEDA before "
                "calling this."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "reason": {
                        "type": "STRING",
                        "description": (
                            "'completed' | 'user_requested_end' | "
                            "'escalated_to_human' | 'no_response' | 'other'"
                        ),
                    },
                },
            },
        ),
    ])
]


def build_system_prompt() -> str:
    """Build the BISVEDA voice assistant system prompt.

    The assistant handles queries from consumers and MSMEs about BIS
    standards, ISI marks, hallmarking, QCO compliance, and more.
    It detects the user's language from their first utterance and
    continues the entire conversation in that language."""

    return """
You are "BISVEDA" (बिसवेदा), a warm, helpful, and knowledgeable AI voice
assistant for the Bureau of Indian Standards (BIS). You are speaking on a
real phone call with someone who needs help with Indian Standards,
product certification, quality marks, or BIS services.

LANGUAGE DETECTION AND HANDLING:
- When the call starts, greet the caller briefly in Hindi first, then in
  English: "Namaste! BISVEDA mein aapka swagat hai. Welcome to BISVEDA."
- Then invite them to speak: "Aap Hindi, English, ya kisi bhi bhasha mein
  apna sawaal pooch sakte hain."
- From the caller's FIRST response, detect their language.
- Continue the ENTIRE rest of the conversation in that detected language.
- If they switch languages mid-call, follow their switch.
- Supported: Hindi, English, Marathi, Tamil, Telugu, Kannada, Bengali,
  Gujarati, Malayalam, Punjabi, and code-mixed (Hinglish).
- Keep sentences SHORT and SIMPLE — many callers have low digital literacy.

YOUR CAPABILITIES (USE THE TOOLS):
1. SEARCH STANDARDS — When the user asks about any BIS standard, product
   standard, or "which standard applies to X", call search_standards with
   English keywords translated from what they said.

2. VERIFY ISI MARK — When the user wants to check an ISI mark / CM-L
   number on a product, ask them to read the number, extract it, and
   call verify_isi_mark. Tell them clearly whether it is valid or not.

3. VERIFY HUID — When the user wants to check a hallmarking HUID on their
   gold/silver jewellery, ask them to read the code, and call verify_huid.
   Report the purity, hallmarking centre, and date.

4. CHECK QCO COMPLIANCE — When the user (typically an MSME/manufacturer)
   asks whether their product needs BIS certification, or whether it is
   under a Quality Control Order, call check_qco_compliance. Explain the
   certification scheme they need to follow.

5. FIND TESTING LABS — When the user needs to find a BIS-recognized lab
   for testing, or an Assaying & Hallmarking Centre (AHC), call
   find_testing_labs with the product/standard and optionally their city.

6. STANDARD PASSPORT — When the user asks for full details about a
   specific known standard (e.g. "Tell me about IS 1239"), call
   get_standard_passport.

7. REPORT COUNTERFEIT — When the user suspects a product has a fake ISI
   mark or HUID, collect: what the product is, where they found it, and
   the number on it, then call report_counterfeit.

8. FILE COMPLAINT — When the user has a quality complaint about a
   BIS-certified product, collect: what the product is, what the issue is,
   and the CM/L number if they have it, then call file_complaint.

CONVERSATIONAL STYLE:
- Be warm, patient, and approachable — like a helpful government service
  desk, not a bureaucrat.
- Ask ONE question at a time and wait for the answer.
- Keep every turn short (1-3 sentences) — this is a phone call.
- When reading out numbers (CM/L numbers, standard numbers), spell them
  clearly digit by digit.
- When presenting search results with multiple items, summarise the top
  result first, then ask if they want to hear more.
- If the user sounds confused, repeat more simply. Never rush them.
- If the user wants to speak to a person, politely tell them this is
  an AI helpline and suggest they contact the nearest BIS regional
  office, then call end_call with reason 'escalated_to_human'.
- After answering a query, ask: "Kya aapka koi aur sawaal hai?" (or
  equivalent in their language). If they say no, thank them warmly,
  say goodbye OUT LOUD, then call end_call.

IMPORTANT RULES:
- NEVER invent or guess standard numbers, QCO statuses, or verification
  results. Always call the appropriate tool and report exactly what it
  returns.
- If a verification fails (ISI mark not found, HUID invalid), tell the
  user clearly and suggest they report it as suspected counterfeit.
- For complex queries beyond your tools (e.g. detailed Form-I filling,
  licence application process), give a brief overview and recommend they
  visit bis.gov.in or contact a BIS regional office.
- Never end the call without first saying goodbye out loud.

When the session starts, greet the caller and invite them to ask their
question in any language.
"""


class GeminiLiveSession:
    def __init__(self, call_id: str, system_prompt: str):
        self.call_id = call_id
        self.system_prompt = system_prompt
        self.session = None
        self._context = None

    async def __aenter__(self):
        self._context = gemini_client.aio.live.connect(
            model=GEMINI_MODEL,
            config=types.LiveConnectConfig(
                response_modalities=["AUDIO"],
                system_instruction=self.system_prompt,
                tools=TOOLS,
                input_audio_transcription=types.AudioTranscriptionConfig(),
                output_audio_transcription=types.AudioTranscriptionConfig(),
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name="Aoede"
                        )
                    )
                ),
            ),
        )
        self.session = await self._context.__aenter__()

        # Kick off the greeting immediately so the caller hears something
        # right away, rather than silence while Gemini waits for audio.
        await self.session.send_client_content(
            turns={
                "role": "user",
                "parts": [{
                    "text": (
                        "The caller has just answered the phone. Greet them "
                        "warmly in Hindi first and then in English as "
                        "instructed, then invite them to ask their question "
                        "in any language."
                    )
                }],
            },
            turn_complete=True,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        return await self._context.__aexit__(exc_type, exc, tb)

    async def send_audio(self, pcm16k: bytes):
        await self.session.send_realtime_input(
            audio=types.Blob(data=pcm16k, mime_type="audio/pcm;rate=16000")
        )

    async def send_tool_response(self, function_responses):
        await self.session.send_tool_response(
            function_responses=function_responses
        )


print("Gemini Live session class + BISVEDA tools ready.")


In [ ]:
# 8. Tool dispatch — executes the function Gemini asked for
#
# Each tool call is wrapped in try/except so one failure never crashes
# the bridge. Results are returned to Gemini as structured dicts.
# Every invocation is also logged to the query_log table.

def execute_tool_call(call_id: str, name: str, args: dict) -> dict:
    state = CALLS.setdefault(call_id, {
        "phone": None, "language": None,
        "stage": "greeting", "should_end": False,
    })

    try:
        if name == "search_standards":
            query = args.get("query", "")
            results = backend_search_standards(query)
            log_query(
                call_id, query, "search_standards", name,
                args, results, f"{len(results)} standards found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"search_standards -> {len(results)} results", call_id)
            return {"status": "ok", "results": results}

        elif name == "verify_isi_mark":
            cm_l = args.get("cm_l_number", "")
            result = backend_verify_isi_mark(cm_l)
            log_query(
                call_id, f"Verify ISI: {cm_l}", "verify_isi_mark", name,
                args, result,
                "Valid" if result.get("valid") else "Invalid/Not found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"verify_isi_mark({cm_l}) -> valid={result.get('valid')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "verify_huid":
            huid = args.get("huid", "")
            result = backend_verify_huid(huid)
            log_query(
                call_id, f"Verify HUID: {huid}", "verify_huid", name,
                args, result,
                "Valid" if result.get("valid") else "Invalid/Not found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"verify_huid({huid}) -> valid={result.get('valid')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "check_qco_compliance":
            product = args.get("product_name", "")
            result = backend_check_qco(product)
            log_query(
                call_id, f"QCO check: {product}", "check_qco", name,
                args, result,
                "Under QCO" if result.get("covered") else "Not under QCO",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"check_qco({product}) -> covered={result.get('covered')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "find_testing_labs":
            product = args.get("product_or_standard", "")
            location = args.get("location")
            results = backend_find_labs(product, location)
            log_query(
                call_id, f"Find labs: {product} near {location}",
                "find_labs", name, args, results,
                f"{len(results)} labs found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"find_testing_labs -> {len(results)} labs", call_id)
            return {"status": "ok", "labs": results}

        elif name == "get_standard_passport":
            std_num = args.get("standard_number", "")
            # Normalize: "IS1239" -> "IS 1239"
            clean = std_num.strip().upper()
            if clean.startswith("IS") and len(clean) > 2 and clean[2].isdigit():
                clean = "IS " + clean[2:]
            result = MOCK_STANDARDS.get(clean, {
                "error": f"Standard '{std_num}' not found in database"
            })
            log_query(
                call_id, f"Standard passport: {std_num}",
                "standard_passport", name, args, result,
                result.get("title", "Not found"),
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"get_standard_passport({clean})", call_id)
            return {"status": "ok", "standard": result}

        elif name == "report_counterfeit":
            report = {
                "report_id": f"CF-{call_id[:8]}-{int(time.time()) % 10000}",
                "product": args.get("product_description"),
                "location": args.get("location"),
                "suspicious_number": args.get("suspicious_number"),
                "details": args.get("additional_details"),
                "status": "logged_for_investigation",
            }
            log_query(
                call_id, f"Counterfeit report: {args.get('product_description')}",
                "report_counterfeit", name, args, report,
                f"Report {report['report_id']} filed",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"report_counterfeit -> {report['report_id']}", call_id)
            return {"status": "ok", "report": report}

        elif name == "file_complaint":
            complaint = {
                "complaint_id": f"CMP-{call_id[:8]}-{int(time.time()) % 10000}",
                "product": args.get("product_description"),
                "details": args.get("complaint_details"),
                "cm_l": args.get("cm_l_number"),
                "purchase_location": args.get("purchase_location"),
                "status": "registered_pending_review",
            }
            log_query(
                call_id, f"Complaint: {args.get('product_description')}",
                "file_complaint", name, args, complaint,
                f"Complaint {complaint['complaint_id']} filed",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"file_complaint -> {complaint['complaint_id']}", call_id)
            return {"status": "ok", "complaint": complaint}

        elif name == "end_call":
            state["should_end"] = True
            reason = args.get("reason", "completed")
            state["end_reason"] = reason
            update_call_session(call_id, end_reason=reason)
            logger.info("CALL_ENDING", f"Agent requested end (reason={reason})", call_id)
            return {"status": "ok"}

        else:
            logger.warn("TOOL_ERROR", f"Unknown tool: {name}", call_id)
            return {"status": "error", "error": f"unknown tool '{name}'"}

    except Exception as e:
        logger.error("TOOL_ERROR", f"Tool '{name}' failed: {type(e).__name__}: {e}", call_id)
        logger.error("TOOL_ERROR_TRACE", traceback.format_exc(), call_id)
        return {"status": "error", "error": f"{type(e).__name__}: {e}"}


print("BISVEDA tool dispatcher ready.")


In [ ]:
# 9. Twilio Media Streams WebSocket bridge — bidirectional audio streaming
#    with Gemini Live function-calling, language detection, and query logging.
#
# The bridge receives audio from Twilio (8kHz mu-law), converts + resamples
# to 16kHz PCM16 for Gemini, and sends Gemini's audio responses (24kHz PCM16)
# back as 8kHz mu-law to Twilio. Tool calls from Gemini are dispatched to
# the BISVEDA backend and results returned to Gemini for spoken response.
#
# Twilio's Media Streams message shapes (all JSON text frames):
#   {"event":"connected", "protocol":"Call", "version":"1.0.0"}
#   {"event":"start", "start": {"callSid":..., "streamSid":..., "mediaFormat":{...}}, "streamSid":...}
#   {"event":"media", "media": {"payload": "<base64 mu-law>"}, "streamSid":...}
#   {"event":"stop", "stop": {"callSid":...}, "streamSid":...}
# and to send audio back:
#   {"event":"media", "streamSid":..., "media": {"payload": "<base64 mu-law>"}}
#   {"event":"clear", "streamSid":...}   <- flushes Twilio's playback buffer (barge-in)

app = FastAPI(title="BISVEDA Voice Bridge")


@app.api_route("/bisveda/twiml", methods=["GET", "POST"])
async def bisveda_twiml(request: Request):
    """Serve TwiML instructing Twilio to open a bidirectional Media Stream to /bisveda/media."""
    stream_url = globals().get("TWILIO_STREAM_URL", "")
    response = VoiceResponse()
    connect = Connect()
    connect.stream(url=stream_url)
    response.append(connect)
    logger.info("TWIML_SERVED", f"Returning stream TwiML for {stream_url}")
    return Response(content=str(response), media_type="application/xml")


@app.get("/health")
def health():
    return {"status": "ok", "service": "BISVEDA AI Voice Assistant", "model": GEMINI_MODEL}


@app.websocket("/bisveda/media")
async def bisveda_media(websocket: WebSocket):
    await websocket.accept()

    call_id = "unknown"
    stream_sid = None
    started_at = time.time()

    system_prompt = build_system_prompt()

    # Counters for the call summary
    incoming_packets = 0
    incoming_bytes = 0
    outgoing_packets = 0
    outgoing_bytes = 0
    gemini_audio_chunks = 0
    user_transcript_chunks = 0
    model_transcript_chunks = 0
    gemini_turns = 0
    tool_calls_handled = 0

    logger.info("CALL_START", "Twilio WebSocket connected", call_id)

    try:
        async with GeminiLiveSession(call_id, system_prompt) as gemini:
            logger.info("GEMINI_CONNECTED", "Gemini Live session connected", call_id)

            # ── Twilio → Gemini (caller audio) ──
            async def twilio_to_gemini():
                nonlocal call_id, stream_sid, incoming_packets, incoming_bytes

                while True:
                    raw = await websocket.receive_text()

                    try:
                        event = json.loads(raw)
                    except json.JSONDecodeError:
                        logger.warn("TWILIO_BAD_FRAME", f"Non-JSON: {raw[:200]}", call_id)
                        continue

                    event_type = event.get("event")

                    if event_type == "connected":
                        logger.info("TWILIO_EVENT", "connected", call_id)

                    elif event_type == "start":
                        start = event.get("start", {})
                        stream_sid = (
                            event.get("streamSid")
                            or start.get("streamSid")
                        )
                        new_call_id = (
                            start.get("callSid")
                            or event.get("callSid")
                            or stream_sid
                            or "bisveda-call"
                        )
                        call_id = new_call_id
                        # Twilio's Stream "start" event doesn't carry the
                        # caller's number by default (unlike Exotel) — we
                        # already know it since call_bisveda() placed the
                        # call, so fall back to MY_PHONE_NUMBER.
                        phone = (
                            (start.get("customParameters") or {}).get("from")
                            or MY_PHONE_NUMBER
                        )
                        CALLS[call_id] = {
                            "phone": phone,
                            "language": None,
                            "stage": "greeting",
                            "should_end": False,
                        }
                        ensure_call_session(call_id, phone)
                        logger.info(
                            "TWILIO_EVENT",
                            f"start | stream_sid={stream_sid} | call_sid={call_id} | "
                            f"format={start.get('mediaFormat')}",
                            call_id,
                        )

                    elif event_type == "media":
                        media = event.get("media", {})
                        payload = media.get("payload")
                        if not payload:
                            logger.warn("TWILIO_EMPTY_MEDIA", "No payload", call_id)
                            continue

                        incoming_packets += 1
                        raw_audio = base64.b64decode(payload)
                        incoming_bytes += len(raw_audio)

                        if incoming_packets == 1:
                            logger.info(
                                "CALLER_AUDIO_START",
                                f"{len(raw_audio)} bytes (mu-law)",
                                call_id,
                            )
                        elif incoming_packets % 100 == 0:
                            logger.info(
                                "CALLER_AUDIO_PROGRESS",
                                f"{incoming_packets} pkts / {incoming_bytes} bytes",
                                call_id,
                            )

                        try:
                            pcm16k = twilio_to_gemini_pcm16(payload, call_id)
                            await gemini.send_audio(pcm16k)
                        except Exception as e:
                            logger.error(
                                "GEMINI_SEND_ERROR",
                                f"{type(e).__name__}: {e}",
                                call_id,
                            )
                            continue

                    elif event_type == "stop":
                        logger.info("TWILIO_STOP", json.dumps(event), call_id)
                        return

                    elif event_type == "mark":
                        # Echoed back for any "mark" messages we sent; not used here.
                        pass

                    else:
                        logger.info("TWILIO_EVENT", str(event_type), call_id)

            # ── Gemini → Twilio (assistant audio + tool calls) ──
            async def gemini_to_twilio():
                nonlocal outgoing_packets, outgoing_bytes, gemini_audio_chunks
                nonlocal user_transcript_chunks, model_transcript_chunks
                nonlocal gemini_turns, tool_calls_handled

                while True:
                    gemini_turns += 1
                    logger.info("GEMINI_WAIT", f"Turn {gemini_turns}", call_id)

                    try:
                        async for response in gemini.session.receive():
                            server_content = getattr(response, "server_content", None)
                            tool_call = getattr(response, "tool_call", None)

                            # ── Function/tool calls ──
                            if tool_call and getattr(tool_call, "function_calls", None):
                                function_responses = []
                                for fc in tool_call.function_calls:
                                    tool_calls_handled += 1
                                    fc_args = dict(fc.args) if fc.args else {}
                                    logger.info(
                                        "TOOL_CALL",
                                        f"{fc.name}({fc_args})",
                                        call_id,
                                    )
                                    result = execute_tool_call(
                                        call_id, fc.name, fc_args
                                    )
                                    function_responses.append(
                                        types.FunctionResponse(
                                            id=fc.id,
                                            name=fc.name,
                                            response=result,
                                        )
                                    )
                                try:
                                    await gemini.send_tool_response(
                                        function_responses
                                    )
                                except Exception as e:
                                    logger.error(
                                        "GEMINI_ERROR",
                                        f"send_tool_response failed: {e}",
                                        call_id,
                                    )

                            if not server_content:
                                continue

                            # ── Barge-in: caller started talking over BISVEDA ──
                            # Twilio keeps playing whatever media frames it has
                            # already been sent, so without this the caller
                            # hears two voices overlap. Gemini flags this via
                            # server_content.interrupted.
                            if getattr(server_content, "interrupted", False):
                                try:
                                    await websocket.send_text(json.dumps({
                                        "event": "clear",
                                        "streamSid": stream_sid,
                                    }))
                                    logger.info(
                                        "BARGE_IN",
                                        "Sent 'clear' to Twilio (caller interrupted)",
                                        call_id,
                                    )
                                except Exception as e:
                                    logger.error(
                                        "TWILIO_SEND_ERROR",
                                        f"clear event failed: {e}",
                                        call_id,
                                    )

                            # ── Caller transcription ──
                            input_tx = getattr(
                                server_content, "input_transcription", None
                            )
                            if input_tx:
                                text = (
                                    getattr(input_tx, "text", None) or ""
                                ).strip()
                                if text:
                                    user_transcript_chunks += 1
                                    logger.info(
                                        "CALLER_TRANSCRIPT", text, call_id
                                    )
                                    append_transcript(call_id, "CALLER", text)

                            # ── Agent transcription ──
                            output_tx = getattr(
                                server_content, "output_transcription", None
                            )
                            if output_tx:
                                text = (
                                    getattr(output_tx, "text", None) or ""
                                ).strip()
                                if text:
                                    model_transcript_chunks += 1
                                    logger.info(
                                        "BISVEDA_TRANSCRIPT", text, call_id
                                    )
                                    append_transcript(call_id, "BISVEDA", text)

                            # ── Audio response ──
                            model_turn = getattr(
                                server_content, "model_turn", None
                            )
                            if model_turn:
                                for part in (
                                    getattr(model_turn, "parts", None) or []
                                ):
                                    inline_data = getattr(
                                        part, "inline_data", None
                                    )
                                    if not inline_data:
                                        continue
                                    audio_data = getattr(
                                        inline_data, "data", None
                                    )
                                    if not audio_data:
                                        continue

                                    gemini_audio_chunks += 1
                                    if gemini_audio_chunks == 1:
                                        logger.info(
                                            "BISVEDA_AUDIO_START",
                                            f"{len(audio_data)} bytes @ 24kHz",
                                            call_id,
                                        )

                                    try:
                                        payload = gemini_to_twilio_payload(
                                            audio_data, call_id
                                        )
                                        await websocket.send_text(
                                            json.dumps({
                                                "event": "media",
                                                "streamSid": stream_sid,
                                                "media": {
                                                    "payload": payload
                                                },
                                            })
                                        )
                                        outgoing_packets += 1
                                        outgoing_bytes += len(payload)
                                    except Exception as e:
                                        logger.error(
                                            "TWILIO_SEND_ERROR",
                                            f"{type(e).__name__}: {e}",
                                            call_id,
                                        )

                            if getattr(
                                server_content, "turn_complete", False
                            ):
                                logger.info(
                                    "GEMINI_TURN_COMPLETE",
                                    "Waiting for caller",
                                    call_id,
                                )

                        logger.info(
                            "GEMINI_ITERATION_END",
                            "Receive iteration ended",
                            call_id,
                        )

                    except Exception as e:
                        logger.error(
                            "GEMINI_ERROR",
                            f"{type(e).__name__}: {e}",
                            call_id,
                        )
                        logger.error(
                            "GEMINI_ERROR_TRACE",
                            traceback.format_exc(),
                            call_id,
                        )
                        await asyncio.sleep(0.5)

                    # Check if agent said goodbye and end the call
                    if CALLS.get(call_id, {}).get("should_end"):
                        logger.info(
                            "CALL_ENDING",
                            "Ending after agent goodbye",
                            call_id,
                        )
                        try:
                            await websocket.close()
                        except Exception:
                            pass
                        return

            # Run both directions concurrently
            input_task = asyncio.create_task(
                twilio_to_gemini(), name="twilio-to-gemini"
            )
            output_task = asyncio.create_task(
                gemini_to_twilio(), name="gemini-to-twilio"
            )

            try:
                done, pending = await asyncio.wait(
                    {input_task, output_task},
                    return_when=asyncio.FIRST_COMPLETED,
                )
                for task in pending:
                    task.cancel()
                    try:
                        await task
                    except (asyncio.CancelledError, Exception):
                        pass
                for task in done:
                    exc = (
                        task.exception()
                        if task.done() and not task.cancelled()
                        else None
                    )
                    if exc:
                        raise exc
            finally:
                pass

    except WebSocketDisconnect:
        logger.warn("WS_DISCONNECT", "Twilio disconnected", call_id)
    except Exception as e:
        logger.error(
            "UNHANDLED_ERROR", f"{type(e).__name__}: {e}", call_id
        )
        logger.error(
            "UNHANDLED_ERROR_TRACE", traceback.format_exc(), call_id
        )
    finally:
        clear_audio_state(call_id)
        duration = time.time() - started_at

        end_reason = CALLS.get(call_id, {}).get("end_reason", "unknown")
        final_status = (
            "completed" if end_reason == "completed"
            else "escalated" if end_reason == "escalated_to_human"
            else "dropped"
        )
        try:
            update_call_session(
                call_id,
                call_ended_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                duration_seconds=round(duration, 1),
                status=final_status,
                end_reason=end_reason,
            )
        except Exception:
            pass

        logger.info(
            "CALL_SUMMARY",
            (
                f"duration={duration:.1f}s | "
                f"in_pkts={incoming_packets} | out_pkts={outgoing_packets} | "
                f"gemini_turns={gemini_turns} | tool_calls={tool_calls_handled} | "
                f"caller_tx={user_transcript_chunks} | agent_tx={model_transcript_chunks} | "
                f"status={final_status}"
            ),
            call_id,
        )
        logger.info(
            "CALL_END",
            f"Full log: {LOG_FILE} | JSONL: {LOG_JSONL_FILE}",
            call_id,
        )


print("WebSocket route registered: /bisveda/media")
print("BISVEDA voice bridge (Twilio) with function calling + barge-in enabled.")


In [ ]:
# 10. Start FastAPI server

import socket
import uvicorn


def _port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(("0.0.0.0", port))
            return True
        except OSError:
            return False


def run_server(port: int):
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning")


if "server_thread" in globals() and server_thread.is_alive():
    print(f"FastAPI already running on port {PORT} (reusing thread).")
else:
    if not _port_is_free(PORT):
        original_port = PORT
        candidate = PORT
        while not _port_is_free(candidate) and candidate < original_port + 20:
            candidate += 1
        print(f"Port {original_port} in use — using {candidate}.")
        PORT = candidate

    server_thread = threading.Thread(
        target=run_server, args=(PORT,), daemon=True
    )
    server_thread.start()
    time.sleep(2)
    print(f"FastAPI running on port {PORT}")


In [ ]:
# 11. Start ngrok tunnel
#
# Twilio needs a public wss:// URL to reach this notebook's WebSocket route,
# same as Exotel did — ngrok still does that job unchanged.

from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)

try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

if public_url.startswith("https://"):
    public_wss_url = "wss://" + public_url[len("https://"):]
elif public_url.startswith("http://"):
    public_wss_url = "wss://" + public_url[len("http://"):]
else:
    public_wss_url = public_url.replace("http://", "wss://")

TWILIO_STREAM_URL = public_wss_url + "/bisveda/media"
TWILIO_TWIML_URL  = public_url + "/bisveda/twiml"

print("=" * 70)
print("NGROK READY")
print("=" * 70)
print("Public HTTPS   :", public_url)
print("Twilio WSS URL :", TWILIO_STREAM_URL)
print("Twilio TwiML   :", TWILIO_TWIML_URL)
print("=" * 70)


In [ ]:
# 12. Place an outbound call (for demo / testing) — via Twilio
#
# call_bisveda(phone_number) places a Twilio outbound call to the given
# number. Twilio fetches TWILIO_TWIML_URL which opens a bidirectional
# Media Stream WebSocket to Gemini Live. When the user picks up, they
# hear the BISVEDA greeting and can converse in Hindi, English, etc.

twilio_client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)


def call_bisveda(phone_number: str = None):
    """Place an outbound BISVEDA call to the given phone number via Twilio."""
    raw_phone = str(phone_number or MY_PHONE_NUMBER).strip()
    # Normalize to E.164
    digits = "".join(c for c in raw_phone if c.isdigit())
    if len(digits) == 10:
        phone = "+91" + digits
    elif len(digits) == 12 and digits.startswith("91"):
        phone = "+" + digits
    elif raw_phone.startswith("+"):
        phone = raw_phone
    else:
        phone = "+91" + digits

    logger.info("TWILIO_CALL_REQUEST", f"to={phone} url={TWILIO_TWIML_URL}")

    try:
        # Use url=TWILIO_TWIML_URL (Twilio trial accounts require url= rather than inline twiml=)
        call = twilio_client.calls.create(
            to=phone,
            from_=TWILIO_PHONE_NUMBER,
            url=TWILIO_TWIML_URL,
        )
        logger.info("TWILIO_CALL_RESPONSE", f"sid={call.sid} status={call.status}")
        print("=" * 70)
        print(f"CALL PLACED SUCCESSFULLY! (SID={call.sid})")
        print(f"To:   {phone}")
        print(f"From: {TWILIO_PHONE_NUMBER} (BISVEDA Official Helpline)")
        print("Please answer your phone — Gemini Live will greet you in seconds!")
        print("Speak naturally in Hindi, English, Marathi, or any Indian language.")
        print("=" * 70)
        return call
    except Exception as e:
        logger.error("TWILIO_CALL_FAILED", f"{type(e).__name__}: {e}")
        # Fallback to inline twiml if trial restrictions change
        try:
            twiml_xml = f'<?xml version="1.0" encoding="UTF-8"?><Response><Connect><Stream url="{TWILIO_STREAM_URL}" /></Connect></Response>'
            call = twilio_client.calls.create(
                to=phone,
                from_=TWILIO_PHONE_NUMBER,
                twiml=twiml_xml,
            )
            return call
        except Exception:
            raise e


print("call_bisveda() ready — run call_bisveda(MY_PHONE_NUMBER) to place your test call!")


In [ ]:
# 13. Place the demo call
# When you answer, try asking:
#   - "Mera ISI mark check karna hai — number hai CM/L-1234567"
#   - "Steel pipes ke liye kaunsa standard lagta hai?"
#   - "Is my gold hallmark real? HUID is AB1234CD"
#   - "Kya cement QCO ke under aata hai?"
#   - "Mumbai mein testing lab batao"

call_bisveda(MY_PHONE_NUMBER)


In [ ]:
# 14. View call sessions and query log (run after a call ends)

import pandas as pd
from IPython.display import display, HTML

sessions = fetch_all_sessions()
queries = fetch_all_queries()

if sessions:
    print(f"\n{'='*60}")
    print(f"  CALL SESSIONS ({len(sessions)})")
    print(f"{'='*60}")
    df_sessions = pd.DataFrame(sessions)
    cols = [
        "call_id", "phone_number", "detected_language",
        "total_queries", "duration_seconds", "status",
    ]
    display(df_sessions[[c for c in cols if c in df_sessions.columns]])
else:
    print("No call sessions yet.")

if queries:
    print(f"\n{'='*60}")
    print(f"  QUERY LOG ({len(queries)})")
    print(f"{'='*60}")
    df_queries = pd.DataFrame(queries)
    cols = [
        "call_id", "timestamp", "detected_intent",
        "tool_called", "response_summary", "language",
    ]
    display(df_queries[[c for c in cols if c in df_queries.columns]])
else:
    print("No queries logged yet.")


In [ ]:
# 15. View and download the call log

from IPython.display import display, FileLink

with open(LOG_FILE, "r", encoding="utf-8") as f:
    log_text = f.read()

print(log_text[-3000:] if len(log_text) > 3000 else log_text)
display(FileLink(LOG_FILE, result_html_prefix="Download text log: "))
display(FileLink(LOG_JSONL_FILE, result_html_prefix="Download JSONL log: "))


In [ ]:
# 16. Analytics — query distribution, language breakdown, tool usage
#
# Run after several calls to see patterns.

import pandas as pd
from IPython.display import display

queries = fetch_all_queries()
sessions = fetch_all_sessions()

if not queries:
    print("No queries yet — run some test calls first.")
else:
    df = pd.DataFrame(queries)

    print("\n" + "=" * 50)
    print("  QUERY DISTRIBUTION BY INTENT")
    print("=" * 50)
    if "detected_intent" in df.columns:
        display(df["detected_intent"].value_counts().to_frame("count"))

    print("\n" + "=" * 50)
    print("  TOOL USAGE")
    print("=" * 50)
    if "tool_called" in df.columns:
        display(df["tool_called"].value_counts().to_frame("count"))

    if sessions:
        df_s = pd.DataFrame(sessions)
        print("\n" + "=" * 50)
        print("  LANGUAGE BREAKDOWN")
        print("=" * 50)
        if "detected_language" in df_s.columns:
            display(
                df_s["detected_language"].value_counts().to_frame("calls")
            )

        print("\n" + "=" * 50)
        print("  CALL OUTCOMES")
        print("=" * 50)
        if "status" in df_s.columns:
            display(df_s["status"].value_counts().to_frame("count"))

        if "duration_seconds" in df_s.columns:
            durations = pd.to_numeric(df_s["duration_seconds"], errors="coerce")
            print(f"\nAvg call duration: {durations.mean():.1f}s")
            print(f"Avg queries/call:  {df_s['total_queries'].astype(float).mean():.1f}")


## How to test end-to-end

```python
# Place an outbound call to your (verified) phone number
call_bisveda(MY_PHONE_NUMBER)
```

When the call connects, try these sample queries:

| Language | Sample query |
|----------|-------------|
| Hindi | "Mera ISI mark check karna hai — number hai CM/L-1234567" |
| Hindi | "Steel pipes ke liye kaunsa standard lagta hai?" |
| Hindi | "Kya cement QCO ke under aata hai?" |
| Hindi | "Mumbai mein testing lab batao" |
| English | "Is my gold hallmark genuine? The HUID is AB1234CD" |
| English | "What is IS 10500?" |
| English | "Do I need BIS certification for electronic toys?" |
| Hinglish | "Mujhe IS 1239 ke baare mein batao" |

The assistant will:
1. Detect your language from your first utterance
2. Continue the conversation in that language
3. Call the appropriate BISVEDA tool (search, verify, check)
4. Speak the results back to you
5. Ask if you have more questions

## Twilio-specific setup checklist

1. **Create a Twilio account** at twilio.com — the trial gives you free credit.
2. **Get a Twilio phone number** (Console -> Phone Numbers -> Buy a number). Any number works since we call out with the REST API, not an inbound webhook.
3. **Verify your own number** under Console -> Phone Numbers -> Verified Caller IDs (required on trial accounts — you can only call verified numbers until you upgrade).
4. **Add secrets in Kaggle** (Add-ons -> Secrets): `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, `TWILIO_PHONE_NUMBER`, `MY_PHONE_NUMBER`, plus your existing `GEMINI_API_KEY` and `NGROK_AUTHTOKEN`.
5. Run the cells top to bottom. On a trial account you'll hear Twilio's short "trial account" message before BISVEDA's greeting — that's expected and disappears after upgrading.

## Production notes

| Concern | Demo (this notebook) | Production |
|---------|---------------------|------------|
| ASR/TTS | Gemini Live native audio | Bhashini ASR + IndicConformer fallback; Bhashini TTS + IndicTTS fallback |
| Standards DB | Mock dict (4 standards) | BISVEDA RAG pipeline over full BIS standards corpus |
| ISI verification | Mock lookup | Real BIS CM/L licence API |
| HUID verification | Mock lookup | Real BIS HUID API |
| QCO database | Mock keyword match | Structured QCO database with gazette notification data |
| Lab directory | Mock list (3 labs) | Full BIS lab directory with geo-filtering |
| Telephony | Twilio trial (free credit) | Twilio paid / Exotel / Jio / BSNL with production VPN/VPC |
| Database | SQLite (single file) | PostgreSQL with replication |
| Tunnel | ngrok (demo) | Direct VPC peering or dedicated endpoint |
| Languages | ~10 (Gemini native) | 22+ scheduled languages via Bhashini |
| Air-gapped | Not supported | IndicConformer (ASR) + IndicTTS (TTS) — fully local |
